In [7]:
from DrissionPage import ChromiumOptions, ChromiumPage
import re
from time import sleep
import pandas as pd
import os
from tqdm import tqdm
import random
from time import sleep
import numpy as np
from DrissionPage.common import Keys, Actions
from datetime import datetime, timedelta

# 手动调试发布时，提前把爆款笔记分析中提炼出的发布话题填在这里。
# 发布代码会在正文末尾逐个输入：#话题 -> sleep(0.5) -> 回车。
PRESELECTED_TOPIC_TAGS = ['数据分析', '数据采集', 'Python爬虫', '公开数据']
PRESELECTED_COLLECTION_NAME = '数据资源类型'
PRESELECTED_COLLECTION_INTRO = '数据采集和数据分析相关内容整理'

def send_cmt():
    judge = 1 if random.random() < 0.2 else 0
    if judge:
        try:
            cmt = get_cmt()
            #print(cmt)

            """激活评论框"""
            page3 = bro.ele('.not-active inner-when-not-active')
            page3.click(by_js=False)
            sleep(1)   

            """输入评论内容"""
            page1 = bro.ele('.content-edit')
            page1.input(cmt)  # 输入
            sleep(1)

            """确认发送"""
            page3 = bro.ele('.btn submit')
            page3.click(by_js=False)
            sleep(1)  
        except:
            pass
        
def shoucang(port,links,cmtt=True):
    
    co = ChromiumOptions().set_paths(local_port=port).mute(True).set_argument('--start-maximized')  # 191本号
    # co = ChromiumOptions().set_paths(local_port=9213).mute(True).set_argument('--start-maximized')  # 151
    # co = ChromiumOptions().set_paths(local_port=9214).mute(True).set_argument('--start-maximized')  # q青青号
    # co = ChromiumOptions().set_paths(local_port=9215).mute(True).set_argument('--start-maximized')  # 大号QQ号
    # co = ChromiumOptions().set_paths(local_port=9216).mute(True).set_argument('--start-maximized')  # 微博号
    #co = ChromiumOptions().set_paths(local_port=9217).mute(True).set_argument('--start-maximized')  # 姐姐号
    


    bro = ChromiumPage(co)
    bro.set.auto_handle_alert()  # 这之后出现的弹窗都会自动确认

    #bro.get(f'https://www.xiaohongshu.com/user/profile/65ec78e0000000000500a01b')
    ac = Actions(bro)
    
    # 获取文本内容
    data2 = []
    beg = 0
    for url in tqdm(links[beg:]):  # 爬取的次数   
        con = []
        try:
            index = links.index(url)
            bro.get(url)
            bro.wait(20,30)
        except:
            continue
        con.append(url)
        """点赞"""
        try:
            like_label = bro.ele('.engage-bar-container').ele('.like-lottie')
            judge_like = bro.ele('.reds-icon like-icon').ele('tag:use').attr('xlink:href')
            if not 'd' in judge_like:
                like_label.click(by_js=None)
        except:
            pass

        """收藏"""
        try:
            like_label = bro.ele('.engage-bar-container').ele('.collect-wrapper')
            judge_like = bro.ele('.reds-icon collect-icon').ele('tag:use').attr('xlink:href')
            if not 'd' in judge_like:
                like_label.click(by_js=None)
        except:
            pass
        if cmtt:
            send_cmt()
    bro.quit()

"""
获取推文+图片
"""
def latest_links(tiao=True):
    data = []
    for m in range(5):  # 爬取的次数     
        try:
            """获取列表元素"""   
            title_elements = bro.s_ele().eles('.note-item')  # class 精确匹配
        except:
            continue

        """开始爬取信息"""
        for title_element in title_elements[:]:

            con = []

            """推文链接"""

            try:
                link = title_element.ele('tag:a').link # 获取src或者href属性值
                #print(link)
                con.append(link)
            except:
                con.append(' ')  

            """推文标题"""
            try:
                textsss =title_element.text
                #print(textsss)
                con.append(textsss)
            except:
                textsss = title_elements.index(title_element)
                con.append(' ')

            """点赞数"""
            try:
                datetime = title_element.ele('.count').text
                #print(datetime)
                con.append(datetime)
            except:
                con.append(' ')

            """作者"""
            try:
                datetimea = title_element.ele('.author-wrapper').text
                #print(datetimea)
                con.append(datetimea)
            except:
                con.append(' ')

            """作者链接"""
            try:
                # 使用xpath获取div元素的class属性(页面元素无此功能)
                datetim =  title_element.ele('.author-wrapper').ele('tag=a').link
                #print(datetim)
                con.append(datetim)
            except:
                con.append(' ')
            judge_like = title_element.ele('.reds-icon like-icon').ele('tag:use').attr('xlink:href')
            if tiao:
                if not 'd' in judge_like:
                    data.append(con)
            else:
                data.append(con)

        """
        记录文件表
        """
        df = pd.DataFrame(data, columns=['链接', '标题', '点赞', '作者', '作者链接'])
        name1 = '链接'
        column_name = name1
        df[column_name] = df[column_name].str.strip()  # 去除首尾空格
        df[column_name] = df[column_name].replace('', pd.NA)  # 将空字符串替换为NaN
        df.dropna(subset=[column_name], inplace=True)
        df.drop_duplicates(subset=column_name, keep='first', inplace=True) # 去重重复值
        #print(f'共获得{len(df[column_name])}条信息',end='\r')
        tuiwen = f'我的小红书推文  {current_date}.xlsx'
        #df.to_excel(tuiwen, index=False)
        links = df['链接'].tolist()
        # df = pd.read_excel(fr"D:\jupyter\实战\爬取网页中的数据\小红书\小红书爬取图片+推文\我的小红书推文  {current_date}.xlsx")
        # biaoti = df['标题'].tolist()
        """
        滚动
        """ 
        for i in range(2):
            #distance, wait_time = generate_random_values()
            bro.scroll.down(1000)
            sleep(3)    
    return links

def all_links(tiao=True):
    data = []
    for m in range(40):  # 爬取的次数     
        try:
            """获取列表元素"""   
            title_elements = bro.s_ele().eles('.note-item')  # class 精确匹配
        except:
            continue

        """开始爬取信息"""
        for title_element in title_elements[:]:

            con = []


            """推文链接"""

            try:
                link = title_element.ele('tag:a').link # 获取src或者href属性值
                #print(link)
                con.append(link)
            except:
                con.append(' ')  

            """推文标题"""
            try:
                textsss =title_element.text
                #print(textsss)
                con.append(textsss)
            except:
                textsss = title_elements.index(title_element)
                con.append(' ')

            """点赞数"""
            try:
                datetime = title_element.ele('.count').text
                #print(datetime)
                con.append(datetime)
            except:
                con.append(' ')

            """作者"""
            try:
                datetimea = title_element.ele('.author-wrapper').text
                #print(datetimea)
                con.append(datetimea)
            except:
                con.append(' ')

            """作者链接"""
            try:
                # 使用xpath获取div元素的class属性(页面元素无此功能)
                datetim =  title_element.ele('.author-wrapper').ele('tag=a').link
                #print(datetim)
                con.append(datetim)
            except:
                con.append(' ')
            judge_like = title_element.ele('.reds-icon like-icon').ele('tag:use').attr('xlink:href')
            if tiao:
                if not 'd' in judge_like:
                    data.append(con)
            else:
                data.append(con)

        """
        记录文件表
        """
        df = pd.DataFrame(data, columns=['链接', '标题', '点赞', '作者', '作者链接'])
        name1 = '链接'
        column_name = name1
        df[column_name] = df[column_name].str.strip()  # 去除首尾空格
        df[column_name] = df[column_name].replace('', pd.NA)  # 将空字符串替换为NaN
        df.dropna(subset=[column_name], inplace=True)
        df.drop_duplicates(subset=column_name, keep='first', inplace=True) # 去重重复值
        print(f'共获得{len(df[column_name])}条信息',end = '\r')
        tuiwen = f'我的小红书推文  {current_date}.xlsx'
        #df.to_excel(tuiwen, index=False)
        """
        滚动
        """ 
        for i in range(2):
            #distance, wait_time = generate_random_values()
            bro.scroll.down(800)
            sleep(2)    

    # df = pd.read_excel(fr"D:\jupyter\实战\爬取网页中的数据\小红书\小红书爬取图片+推文\我的小红书推文  {current_date}.xlsx")
    links = df['链接'].tolist()
    # biaoti = df['标题'].tolist()
    return links

def clean_file_name(file_name):
    # 定义非法字符集合,包含所有不允许出现在文件名中的字符
    invalid_chars = r'[<>:/\\|?*"]'
    # 使用正则表达式替换所有非法字符为空字符串
    cleaned_file_name = re.sub(invalid_chars, '', file_name)
    return cleaned_file_name

def generate_time_slots():
    time_slots = [
    "6:30", "7:30", "8:30",
    "12:00", "13:00", "13:30",
    "19:00", "19:30", "20:00",
    "20:30", "21:00", "21:30",
    "22:30", "23:00"
    ]

    current_time = datetime.now()
    start_date = current_time.strftime("%Y-%m-%d %H:%M")

    current_time = datetime.now()
    
    # 将开始时间字符串转换为时间对象
    start_time = datetime.strptime(start_date, "%Y-%m-%d %H:%M")
    
    # 如果开始时间比当前时间早,则从当前时间开始
    if start_time < current_time:
        start_time = current_time
    
    # 获取一周后的日期
    end_time = start_time + timedelta(days=3)
    
    # 生成时间段列表
    time_slot_list = []
    current_date = start_time.date()
    
    while current_date <= end_time.date():
        for slot in time_slots:
            slot_time = datetime.strptime(slot, "%H:%M").time()
            if current_date == current_time.date() and slot_time <= current_time.time():
                continue
            date_str = current_date.strftime("%Y-%m-%d")
            time_str = slot
            time_slot_list.append(date_str + " " + time_str)
        current_date += timedelta(days=1)
    
    return time_slot_list

def get_text():
    df = pd.read_excel(r"/Users/wangbin/data/jupyter/实战/爬网页/小红书/自动发推文/merged_table.xlsx")
    biaoti = df['标题'].tolist()
    tuiwen = df['推文'].tolist()
    fengmian = df['封面地址'].tolist()
    index = random.randint(0, len(biaoti))

    return biaoti[index], tuiwen[index], fengmian[index]

def generate_random_subset(input_list,beg,end):
    new_length = random.randint(beg,end)
    new_list = random.sample(input_list, new_length)
    return new_list

"""点击图文"""
def tuwen():
    try:
        bro.get(f'https://creator.xiaohongshu.com/publish/publish?from=menu&target=image')
    except:
        bro.get(f'https://creator.xiaohongshu.com/publish/publish')
    try:
        page = bro.ele('text=上传图文')
        ac.move_to(ele_or_loc=page).click()
    except:
        pass


"""话题"""
def openq():
    try: 
        page = bro.ele('#topicBtn')
        # 如元素不被遮挡,用模拟点击,否则用js点击
        
        bro.scroll.down(600)
        
        page.click(by_js= None)
        
        #ac.move_to(ele_or_loc=page).click()
        sleep(2)
    except:
        pass

def gethuatis():

    bro.wait(3)
    con = []
    try:
        title_elements = bro.ele('#quill-mention-list').children('t:li')
        for title_element in title_elements:
            name1 = """店名"""
            try:
                hotelName = title_element.ele('.item-name').text
                #print(hotelName)
                con.append(hotelName)
            except:
                
                continue
        return con
    
    except:
        return [1,2,3,4,5,6]
        pass
    
def xuan(con):
    for huati in range(len(con)):
        if huati!=10 and huati!=20:
            openq()

            try: 
                title_elementss = bro.ele('#quill-mention-list').child('t:li',huati+1)
                # 如元素不被遮挡,用模拟点击,否则用js点击
                ac.move_to(ele_or_loc=title_elementss ).click()
                #ac.move_to(ele_or_loc=page).click()
                sleep(1)
            except:
                pass
        else:
            continue

def normalize_topic_tag(raw):
    return str(raw or '').strip().lstrip('#').strip()


def input_preselected_topic_tags(input_target, topic_tags):
    for raw_tag in topic_tags:
        tag = normalize_topic_tag(raw_tag)
        if not tag:
            continue
        input_target.input(f'#{tag}')
        sleep(0.5)
        input_target.input('\n')


def xuanhuati(input_target, topic_tags=None):
    # 新版必须提前确定话题词后，在正文编辑框末尾逐个输入。
    input_preselected_topic_tags(input_target, topic_tags or PRESELECTED_TOPIC_TAGS)

def normalize_collection_text(value):
    return re.sub(r'\s+', '', str(value or '').strip())


def parse_collection_names(raw_text):
    names = []
    for line in str(raw_text or '').splitlines():
        name = line.strip()
        if not name or '创建合集' in name or name in {'合集', '选择合集'}:
            continue
        if name not in names:
            names.append(name)
    return names


def collection_match_score(target, existing, topic_tags):
    target_norm = normalize_collection_text(target)
    existing_norm = normalize_collection_text(existing)
    if not target_norm or not existing_norm:
        return 0
    if target_norm == existing_norm:
        return 100
    if target_norm in existing_norm or existing_norm in target_norm:
        return 80
    score = 0
    for tag in topic_tags:
        tag_norm = normalize_collection_text(tag)
        if tag_norm and (tag_norm in existing_norm or existing_norm in tag_norm):
            score += 20
    for keyword in ('数据', '爬虫', '采集', '公开', '分析', '塔罗', '旅行', '拍照', '国风', '搞笑'):
        if keyword in target_norm and keyword in existing_norm:
            score += 15
    return score


def best_existing_collection(target, existing_names, topic_tags):
    scored = [(collection_match_score(target, name, topic_tags), name) for name in existing_names]
    scored = [(score, name) for score, name in scored if score >= 20]
    if not scored:
        return None
    scored.sort(key=lambda item: (-item[0], len(item[1])))
    return scored[0][1]


def select_or_create_collection(collection_name=None, collection_intro=None, topic_tags=None):
    topic_tags = topic_tags or PRESELECTED_TOPIC_TAGS
    collection_name = (collection_name or PRESELECTED_COLLECTION_NAME)[:20]
    collection_intro = (collection_intro or PRESELECTED_COLLECTION_INTRO)[:50]

    sleep(3)
    page4 = bro.ele('.collection-plugin-wrapper')
    ac.move_to(ele_or_loc=page4).click()
    sleep(1)

    popover = bro.ele('.collection-plugin-popover-content')
    existing_text = popover.text
    print(existing_text)
    target_collection = best_existing_collection(collection_name, parse_collection_names(existing_text), topic_tags)

    if target_collection:
        page4 = popover.ele(f'tx:{target_collection}')
        ac.move_to(ele_or_loc=page4).click()
        sleep(1)
        selected = bro.ele('.collection-name').text
        print(selected)
        if normalize_collection_text(target_collection) not in normalize_collection_text(selected):
            raise RuntimeError(f'合集选择后校验不一致：期望 {target_collection}，页面显示 {selected}')
        return

    page4 = popover.ele('tx= 创建合集 ')
    ac.move_to(ele_or_loc=page4).click()

    page4 = bro.ele('@placeholder=好的合集名称能吸引更多用户')
    page4.input(collection_name)

    page4 = bro.ele('@placeholder=简单介绍你的合集')
    page4.input(collection_intro)

    page4 = bro.ele('tx=创建并加入')
    ac.move_to(ele_or_loc=page4).click()
    sleep(3)
    selected = bro.ele('.collection-name').text
    print(selected)
    if normalize_collection_text(collection_name) not in normalize_collection_text(selected):
        raise RuntimeError(f'合集创建后校验不一致：期望 {collection_name}，页面显示 {selected}')


def dingshi(numm=30):
    
    # 指定文件夹路径
    path1 = r"/Users/wangbin/data/jupyter/实战/爬网页/小红书/自动发推文/封面"
    path = r"/Users/wangbin/data/jupyter/实战/爬网页/小红书/自动发推文/封面20250120"

    pic_paths = []

    for file in os.listdir(path):
        if file.endswith('.jpg') or file.endswith('.png'):
            pic_paths.append(os.path.join(path, file))

    pic_paths1 = []

    for file in os.listdir(path1):
        if file.endswith('.jpg') or file.endswith('.png'):
            pic_paths1.append(os.path.join(path1, file))
            
    time_slot_list = generate_time_slots()

    for time22 in tqdm(time_slot_list[:numm]):
        try:
            tuwen()
            """开始上传内容"""
            pic_lis = generate_random_subset(pic_paths,3,5)
            pic_lis1 = generate_random_subset(pic_paths1,1,2)
            _biaoti, _tuiwen, _fengm = get_text()

            #tuiwen = tuiwen + '\n有意向可以加我WeChat:w15117119070(备注来意)'
            """上传图片"""
            pageg = bro.ele('.creator-container').ele('tx=上传图片')
            #for filelp in pic_lis[:1]:
            #pageg.click.to_upload(pic_lis1[0])
            
            pageg.click.to_upload(_fengm)
            
            
            bro.wait(2,3)
            pageg = bro.ele('.entry')
            for filelp in pic_lis[:]:
                pageg.click.to_upload(filelp)
                
            pageg.click.to_upload(r"/Users/wangbin/data/jupyter/实战/总结/1.png")
            pageg.click.to_upload(r"/Users/wangbin/data/jupyter/实战/总结/2.png")
            pageg.click.to_upload(r"/Users/wangbin/data/jupyter/实战/总结/3.png")
            
            """输入标题"""
            page1 = bro.ele('@placeholder=填写标题会有更多赞哦～')
            page1.input(_biaoti[:19]+'!')  # 输入
            sleep(1)
            """输入详细文章"""
            page2 = bro.ele('@data-placeholder=输入正文描述，真诚有价值的分享予人温暖')
            buchong = """
            \n
            😋最后3张图是我做过的项目详情页，超多干货文字放不下，具体服务、数据表字段都在图里啦，冲鸭🌟

            💡温馨提示：所有数据采集均基于公开可访问信息，严格遵守《中华人民共和国网络安全法》《中华人民共和国数据保护法》及各平台服务条款、隐私政策，合法合规做服务，放心冲就对啦✨
            \n
            """
            wenzhang = _tuiwen.split('#')[0] + buchong
            page2.input(wenzhang[:900])

            
            page2.input('\n')
            ac.key_down(Keys.END).key_up(Keys.END)
            sleep(1)

            """选话题：提前确定话题词，逐个输入 #话题 -> 0.5秒 -> 回车"""
            xuanhuati(page2, PRESELECTED_TOPIC_TAGS)

            """选择或创建合集：放在原创声明之后、定时发布之前"""
            select_or_create_collection(PRESELECTED_COLLECTION_NAME, PRESELECTED_COLLECTION_INTRO, PRESELECTED_TOPIC_TAGS)
            """定时发布"""
            page3 = bro.ele('text=定时发布')
            page3.click(by_js=None)
            sleep(1)    


            page4 = bro.ele('.el-input__prefix-inner')
            page4.input('1')  # 输入
            for i in range(20):
                ac.move_to(ele_or_loc=page4).key_down(Keys.END).key_up(Keys.END).key_down(Keys.BACK_SPACE).key_up(Keys.BACK_SPACE)
                sleep(0.2)

            page4.input(time22)  # 输入

            #ac.key_down(Keys.ESCAPE).key_up(Keys.ESCAPE)
            sleep(1)    

            """确认发布"""
            page5 = bro.ele('.:publishBtn')
            ac.move_to(ele_or_loc=page5).click()
            bro.wait(2,3)
            # 检查是否发布成功
            try:
                fabuchengg = bro.ele('text=发布成功').text
                print(fabuchengg)
            except:
                pass
        except:
            pass

def jishi(num):
    
    # 指定文件夹路径
    path = r"/Users/wangbin/data/jupyter/实战/爬网页/小红书/自动发推文/封面20240608"
    pic_paths = []

    for file in os.listdir(path):
        if file.endswith('.jpg') or file.endswith('.png'):
            pic_paths.append(os.path.join(path, file))
            
    time_slot_list = generate_time_slots()

    for time22 in tqdm(time_slot_list[:num]):
        try:
            tuwen()
            """开始上传内容"""
            pic_lis = generate_random_subset(pic_paths)
            _biaoti, _tuiwen = get_text()

            #tuiwen = tuiwen + '\n有意向可以加我WeChat:w15117119070(备注来意)'
            """上传图片"""
            pageg = bro.ele('.upload-wrapper').ele('.upload-input')
            for filelp in pic_lis[:1]:
                pageg.click.to_upload(filelp)
            

            bro.wait(2,3)
            pageg = bro.ele('.entry')
            for filelp in pic_lis[1:]:
                pageg.click.to_upload(filelp)
                
            pageg.click.to_upload(r"/Users/wangbin/data/jupyter/实战/总结/1.png")
            pageg.click.to_upload(r"/Users/wangbin/data/jupyter/实战/总结/2.png")
            pageg.click.to_upload(r"/Users/wangbin/data/jupyter/实战/总结/3.png")
            
            """输入标题"""
            page1 = bro.ele('@placeholder=填写标题会有更多赞哦～')
            page1.input(_biaoti)  # 输入
            sleep(1)
            """输入详细文章"""
            page2 = bro.ele('@data-placeholder=输入正文描述，真诚有价值的分享予人温暖')
            buchong = """
            \n图片的最后一张是我做过的项目目录，如果有你感兴趣的直接联系我要项目目录详情，里面有更明确的各数据表的字段详情喔，相中的话价格一律30起

            爬虫仅会收集那些公开可访问的数据，爬虫严格遵守所在国家及地区的法律法规，包括但不限于《中华人民共和国网络安全法》、《中华人民共和国数据保护法》以及各网站的服务条款与隐私政策。~~~
            电商平台#淘宝#京东#唯品会#亚马逊商品信息#商品详情#商品参数#SKU#销量#价格#店铺信息用户反馈#评论#评分#用户ID#日期网页新闻#新闻平台#公众号#搜索引擎(百度, 必应, 谷歌)#新闻网站(人民网, 中国日报, 头条, 搜狐网)#外文新闻#纽约时报#chinadaily#数据类型#标题#链接#发布日期#全文#AI总结#批量操作#图片下载#文档下载(PDF, Word)地图数据#地理信息#地图#地址#电话#链接主流媒体平台数据#短视频平台#抖音#快手#TikTok#YouTube#社交平台#微博#小红书#推特#Instagram#内容类型#视频#图片#评论#博主信息#粉丝数据#弹幕#互动行为#点赞#评论#分享#收藏#转发数据分析#文本分析#情感分析#词频#LDA主题分析#网络图分析#统计分析#时间分布#类别饼图#统计图表机器学习#算法#回归#随机森林#XGBoost#LightGBM#BP神经网络#决策树#岭回归#多项式回归#遗传算法#应用领域#图像识别#分类#预测公开数据服务#学术资源#期刊信息#娱乐体育#竞彩网站#财经金融#股票网站#其他公开数据合法声明#法律法规#网络安全法#数据保护法#合规要求#robots.txt#
            """
            wenzhang = _tuiwen + buchong[:930]
            page2.input(wenzhang)
            sleep(1)    

            
            #ac.key_down(Keys.ESCAPE).key_up(Keys.ESCAPE)
            sleep(1)    

            """确认发布"""
            page5 = bro.ele('.:publishBtn')
            ac.move_to(ele_or_loc=page5).click()
            bro.wait(2,3)
            # 检查是否发布成功
            try:
                fabuchengg = bro.ele('text=发布成功').text
                print(fabuchengg)
            except:
                pass
        except:
            pass

def get_cmt():
    df = pd.read_excel("/Users/wangbin/data/jupyter/实战/爬网页/小红书/自动发推文/评论.xlsx")
    biaoti = df['评论'].tolist()

    index = random.randint(0, len(biaoti))
    return biaoti[index]

def main():
    current_date = datetime.now().strftime('%Y-%m-%d %H-%M-%S')

    print(current_date)
    #sleep(1200)
    # 获取当前日期
    current_date = datetime.now().strftime('%Y-%m-%d')
    co = ChromiumOptions().set_paths(local_port=9212).mute(True).set_argument('--start-maximized')
    bro = ChromiumPage(co)
    bro.set.auto_handle_alert()  # 这之后出现的弹窗都会自动确认
    bro.get(f'https://creator.xiaohongshu.com/publish/publish')
    ac = Actions(bro)
    #sleep(600)
    dingshi()
    #jishi()
    bro.quit()

    for port in np.arange(9212,9216):
        #         if port == 9211 or port == 9213:
        #             continue
        #print(port)
        current_date = datetime.now().strftime('%Y-%m-%d')
        co = ChromiumOptions().set_paths(local_port=port).mute(True).set_argument('--start-maximized')
        bro = ChromiumPage(co)
        bro.set.auto_handle_alert()  # 这之后出现的弹窗都会自动确认
        #bro.get(f'https://creator.xiaohongshu.com/publish/publish')
        ac = Actions(bro)

        bro.get(f'https://www.xiaohongshu.com/user/profile/65ec78e0000000000500a01b')  # qq号
        try:
            links = latest_links()
            print(f'{port} 共获得{len(links)}条推文')
            #dianzan(port)
            shoucang(port,links,cmtt=False)
        except Exception as e:
            print(f'{port}     {e}')



In [49]:
# sleep(3600)
# 获取当前日期

current_date = datetime.now().strftime('%Y-%m-%d')
co = ChromiumOptions().set_paths(local_port=9209).mute(True).set_argument('--start-maximized').ignore_certificate_errors(True)
bro = ChromiumPage(co)
bro.set.auto_handle_alert()  # 这之后出现的弹窗都会自动确认
ac = Actions(bro)       
bro.get(f'https://creator.xiaohongshu.com/publish/publish?from=menu')
numm = 1
# 指定文件夹路径
path1 = r"/Users/wangbin/data/jupyter/实战/爬网页/小红书/自动发推文/封面"
path = r"/Users/wangbin/data/jupyter/实战/爬网页/小红书/自动发推文/封面20250120"

pic_paths = []

for file in os.listdir(path):
    if file.endswith('.jpg') or file.endswith('.png'):
        pic_paths.append(os.path.join(path, file))

pic_paths1 = []

for file in os.listdir(path1):
    if file.endswith('.jpg') or file.endswith('.png'):
        pic_paths1.append(os.path.join(path1, file))
        
time_slot_list = generate_time_slots()

for time22 in tqdm(time_slot_list[:numm]):
    try:
        try:
            bro.get(f'https://creator.xiaohongshu.com/publish/publish?from=menu&target=image')
        except:
            bro.get(f'https://creator.xiaohongshu.com/publish/publish')
        try:
            page = bro.ele('text=上传图文')
            ac.move_to(ele_or_loc=page).click()
        except:
            pass

        """开始上传内容"""
        pic_lis = generate_random_subset(pic_paths,3,5)
        pic_lis1 = generate_random_subset(pic_paths1,1,2)
        _biaoti, _tuiwen, _fengm = get_text()

        #tuiwen = tuiwen + '\n有意向可以加我WeChat:w15117119070(备注来意)'
        """上传图片"""
        pageg = bro.ele('tx=上传图片')
        #for filelp in pic_lis[:1]:
        #pageg.click.to_upload(pic_lis1[0])
        
        pageg.click.to_upload(_fengm)
        
        
        bro.wait(2,3)
        pageg = bro.ele('.entry')
        for filelp in pic_lis[:]:
            pageg.click.to_upload(filelp)
            
        pageg.click.to_upload(r"/Users/wangbin/data/jupyter/实战/总结/1.png")
        pageg.click.to_upload(r"/Users/wangbin/data/jupyter/实战/总结/2.png")
        pageg.click.to_upload(r"/Users/wangbin/data/jupyter/实战/总结/3.png")
        
        """输入标题"""
        page1 = bro.ele('@placeholder:填写标题')
        page1.input(_biaoti[:19]+'!')  # 输入
        sleep(1)

        """输入详细文章"""
        page2 = bro.ele('@data-placeholder:输入正文描述')
        buchong = """
        \n
        😋最后3张图是我做过的项目详情页，超多干货文字放不下，具体服务、数据表字段都在图里啦，冲鸭🌟

        💡温馨提示：所有数据采集均基于公开可访问信息，严格遵守《中华人民共和国网络安全法》《中华人民共和国数据保护法》及各平台服务条款、隐私政策，合法合规做服务，放心冲就对啦✨
        \n
        """
        wenzhang = _tuiwen.split('#')[0] + buchong
        page2.input(wenzhang[:900])

        
        page2.input('\n')
        ac.key_down(Keys.END).key_up(Keys.END)
        sleep(1)

        """选话题：提前确定话题词，逐个输入 #话题 -> 0.5秒 -> 回车"""
        xuanhuati(page2, PRESELECTED_TOPIC_TAGS)

        """选择或创建合集：放在原创声明之后、定时发布之前"""
        select_or_create_collection(PRESELECTED_COLLECTION_NAME, PRESELECTED_COLLECTION_INTRO, PRESELECTED_TOPIC_TAGS)
        # 在定时发布之前，让屏幕滚动500
        from random import randint, uniform

        sleep(3)

        size = bro.run_js('return {w: window.innerWidth, h: window.innerHeight};')
        center_x = size['w'] // 2
        center_y = size['h'] // 2

        ac.move_to((center_x, center_y))

        total = 0
        while total < 700:
            step = randint(35, 70)
            ac.scroll(delta_y=step)
            total += step
            sleep(uniform(0.08, 0.18))

        """定时发布"""
        page3 = bro.ele('.custom-switch-switch',index=-1).ele('.d-switch-box')
        page3.click(by_js=None)
        sleep(1)    

        time22 = "2026-05-10 13:00"

        sleep(3)
        page4 = bro.ele('.d-datepicker-suffix --color-text-description d-datepicker-suffix-indicator')
        print(time22)
        # print(page4)
        # page4.click(by_js=None)
        ac.move_to(ele_or_loc=page4).click()

        ac.key_down(Keys.SHIFT)

        for i in range(20):
            ac.key_down(Keys.LEFT).key_up(Keys.LEFT)
            sleep(0.05)

        ac.key_up(Keys.SHIFT)

        page4.input(time22)



        """确认发布"""
        # page5 = bro.ele('tx=定时发布',index=-1)
        # ac.move_to(ele_or_loc=page5).click()
        # bro.wait(2,3)
    except:
        pass
# bro.quit()           

  0%|          | 0/1 [00:00<?, ?it/s]

2026-05-10 13:00


100%|██████████| 1/1 [00:50<00:00, 50.49s/it]
